In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('results/application', exist_ok=True)
print('Working directory:', os.getcwd())

Working directory: /Users/kerie/Downloads/treesight_notebooks


# Notebook 05 — Predict On A Real Rwanda Parcel
## Test your trained model on any Rwanda land parcel
This simulates what happens when a citizen opens the app and enters their UPI.

In [2]:
import pickle
import numpy as np
import pandas as pd
import folium

print('Libraries loaded')

Libraries loaded


## Step 1 — Load Your Trained Model
We use the best model (Experiment D — all sources combined)

In [3]:
with open('models/rf_D.pkl', 'rb') as f:
    model = pickle.load(f)

print('Model loaded')
print(f'Number of trees: {model.n_estimators}')
print(f'Features the model knows: {len(model.feature_importances_)}')

Model loaded
Number of trees: 400
Features the model knows: 17


## Step 2 — Define The Prediction Function
This is what runs when a citizen selects their parcel in the app

In [4]:
def predict_parcel(parcel_data, model, feature_names):
    """Predict deforestation risk for a single land parcel."""
    X = np.array([[parcel_data.get(f, 0) for f in feature_names]])
    prob_deforested = model.predict_proba(X)[0][1]
    predicted_class = model.predict(X)[0]
    ndvi_now = parcel_data.get('NDVI_test', parcel_data.get('NDVI_train', 0.5))
    ndvi_before = parcel_data.get('NDVI_train', ndvi_now)
    tree_cover_now = max(0, min(100, ndvi_now * 120))
    tree_cover_before = max(0, min(100, ndvi_before * 120))
    tree_cover_change = tree_cover_before - tree_cover_now
    if prob_deforested > 0.65 or tree_cover_change > 20:
        risk = 'HIGH'
    elif prob_deforested > 0.35 or tree_cover_change > 10:
        risk = 'MEDIUM'
    else:
        risk = 'LOW'
    return {
        'deforestation_probability': round(prob_deforested * 100, 1),
        'risk_level': risk,
        'tree_cover_2020_pct': round(tree_cover_before, 1),
        'tree_cover_2024_pct': round(tree_cover_now, 1),
        'tree_cover_change_pct': round(tree_cover_change, 1),
        'model_prediction': 'Deforested' if predicted_class == 1 else 'Stable Forest'
    }

print('Prediction function defined')

Prediction function defined


## Step 3 — Test On Example Rwanda Parcels
These are realistic pixel values for Nyungwe buffer zone parcels

In [5]:
df = pd.read_csv('data/processed/training_data_clean.csv')
feature_names = [c for c in df.columns if c != 'label']

test_parcels = [
    {'name': 'Parcel A - Recently cleared (HIGH expected)', 'location': (-2.65, 29.12),
     'data': {'NDVI_train':0.78,'NDVI_test':0.18,'NDVI_change':-0.60,'EVI_train':0.65,
              'SWIR_train':0.12,'SWIR_test':0.48,'NBR_train':0.52,'VH_train':-12.3,
              'VV_train':-9.1,'VH_test':-21.4,'VV_test':-16.2,'VH_VV_ratio':2.35,
              'elevation':1847,'slope':12.3,'aspect':245.0}},
    {'name': 'Parcel B - Stable forest (LOW expected)', 'location': (-2.72, 29.05),
     'data': {'NDVI_train':0.82,'NDVI_test':0.80,'NDVI_change':-0.02,'EVI_train':0.71,
              'SWIR_train':0.08,'SWIR_test':0.09,'NBR_train':0.62,'VH_train':-11.8,
              'VV_train':-8.7,'VH_test':-12.1,'VV_test':-8.9,'VH_VV_ratio':1.35,
              'elevation':2103,'slope':22.1,'aspect':182.0}},
    {'name': 'Parcel C - Partial clearing (MEDIUM expected)', 'location': (-2.58, 29.20),
     'data': {'NDVI_train':0.74,'NDVI_test':0.52,'NDVI_change':-0.22,'EVI_train':0.61,
              'SWIR_train':0.14,'SWIR_test':0.28,'NBR_train':0.48,'VH_train':-13.1,
              'VV_train':-9.8,'VH_test':-16.2,'VV_test':-12.1,'VH_VV_ratio':1.72,
              'elevation':1654,'slope':8.4,'aspect':120.0}},
]

print('=== PARCEL PREDICTION RESULTS ===\n')
for parcel in test_parcels:
    result = predict_parcel(parcel['data'], model, feature_names)
    print(f"Parcel: {parcel['name']}")
    print(f"  Risk Level:         {result['risk_level']}")
    print(f"  Deforestation prob: {result['deforestation_probability']}%")
    print(f"  Tree cover 2020:    {result['tree_cover_2020_pct']}%")
    print(f"  Tree cover 2024:    {result['tree_cover_2024_pct']}%")
    print(f"  Tree cover lost:    {result['tree_cover_change_pct']}%")
    print(f"  Model says:         {result['model_prediction']}")
    print()

=== PARCEL PREDICTION RESULTS ===

Parcel: Parcel A - Recently cleared (HIGH expected)
  Risk Level:         HIGH
  Deforestation prob: 43.8%
  Tree cover 2020:    93.6%
  Tree cover 2024:    21.6%
  Tree cover lost:    72.0%
  Model says:         Stable Forest

Parcel: Parcel B - Stable forest (LOW expected)
  Risk Level:         MEDIUM
  Deforestation prob: 35.8%
  Tree cover 2020:    98.4%
  Tree cover 2024:    96.0%
  Tree cover lost:    2.4%
  Model says:         Stable Forest

Parcel: Parcel C - Partial clearing (MEDIUM expected)
  Risk Level:         HIGH
  Deforestation prob: 52.7%
  Tree cover 2020:    88.8%
  Tree cover 2024:    62.4%
  Tree cover lost:    26.4%
  Model says:         Deforested



## Step 4 — Show Results On Interactive Rwanda Map

In [6]:
m = folium.Map(location=[-2.65, 29.12], zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri')

color_map = {'HIGH':'red','MEDIUM':'orange','LOW':'green'}
for parcel in test_parcels:
    result = predict_parcel(parcel['data'], model, feature_names)
    color = color_map[result['risk_level']]
    popup_html = f"<b>{result['risk_level']} RISK</b><br>Tree cover 2020: {result['tree_cover_2020_pct']}%<br>Tree cover 2024: {result['tree_cover_2024_pct']}%<br>Prob: {result['deforestation_probability']}%"
    folium.CircleMarker(location=parcel['location'], radius=12, color=color,
        fill=True, fill_color=color, fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=parcel['name'].split(' - ')[0]).add_to(m)

map_path = 'results/application/parcel_predictions_map.html'
m.save(map_path)
print(f'Interactive map saved to: {map_path}')
print('Red = HIGH, Orange = MEDIUM, Green = LOW')
m

Interactive map saved to: results/application/parcel_predictions_map.html
Red = HIGH, Orange = MEDIUM, Green = LOW


## Step 5 — Alternative Suggestions Based On Risk

In [7]:
def get_alternatives(reason_for_cutting, risk_level):
    """Return pre-written alternatives based on reason and risk level."""
    alternatives = {
        'firewood': ['Improved cookstoves - REG subsidy available',
                     'Household biogas digesters - government subsidy',
                     'Dead wood harvesting - no permit required'],
        'timber': ['Bamboo planting - poles in 3-4 years',
                   'Phased cutting - cut 30% now, wait 2 years',
                   'Dead/fallen timber harvesting'],
        'farming': ['Agroforestry - crops between trees',
                    'Intercropping - RFA/DeSIRA support',
                    'Terraced farming on existing cleared land'],
        'income': ['Carbon credits - 2024 Forest Law enables selling',
                   'Ecotourism - Nyungwe tourism programs',
                   'REMA conservation payments - rema.gov.rw'],
    }
    result = alternatives.get(reason_for_cutting.lower(), ['Contact RFA: rfa.rw'])
    label = {'HIGH':'HIGH RISK - Strongly recommend alternatives:',
             'MEDIUM':'MEDIUM RISK - Consider these alternatives:',
             'LOW':'LOW RISK - These may still be worth considering:'}
    print(label.get(risk_level, ''))
    for i, alt in enumerate(result, 1):
        print(f'  {i}. {alt}')

print('--- firewood, HIGH RISK ---')
get_alternatives('firewood', 'HIGH')
print()
print('--- farming, MEDIUM RISK ---')
get_alternatives('farming', 'MEDIUM')

--- firewood, HIGH RISK ---
HIGH RISK - Strongly recommend alternatives:
  1. Improved cookstoves - REG subsidy available
  2. Household biogas digesters - government subsidy
  3. Dead wood harvesting - no permit required

--- farming, MEDIUM RISK ---
MEDIUM RISK - Consider these alternatives:
  1. Agroforestry - crops between trees
  2. Intercropping - RFA/DeSIRA support
  3. Terraced farming on existing cleared land


## Congratulations — Your Model Works End To End

In [8]:
print('='*55)
print('YOUR TREESIGHT MODEL PIPELINE IS WORKING')
print('='*55)
print()
print('What you have now:')
print('  models/rf_A..D.pkl              - four trained models (D best)')
print('  results/experiments/experiment_results.csv  - F1 scores')
print('  results/experiments/feature_importance.png  - top features (RQ1)')
print('  results/metrics/confusion_matrix.png        - true vs false detections')
print('  results/application/parcel_predictions_map.html - interactive map')
print()
print('Next steps:')
print('  1. Use results in research report chapters 3 and 4')
print('  2. Web app (app_cadastral.py) serves the model live')
print('  3. Deploy backend on Render (gunicorn) for supervisor demo')

YOUR TREESIGHT MODEL PIPELINE IS WORKING

What you have now:
  models/rf_A..D.pkl              - four trained models (D best)
  results/experiments/experiment_results.csv  - F1 scores
  results/experiments/feature_importance.png  - top features (RQ1)
  results/metrics/confusion_matrix.png        - true vs false detections
  results/application/parcel_predictions_map.html - interactive map

Next steps:
  1. Use results in research report chapters 3 and 4
  2. Web app (app_cadastral.py) serves the model live
  3. Deploy backend on Render (gunicorn) for supervisor demo
